# VARN-LQG on the bench (DAO)

The controller of `RamaCup/rama_controller/varn_lqg`, run from this notebook on RAMA, with the shared memories of `DAOKalman.ipynb`. `VarNLQGTutorial.ipynb` explains each step in simulation. Here the same steps run on the bench:

1. **Delay and optical gain.** A small random dither on the integrator's commands. The lag at which it shows up in `z` gives the loop delay (fractional allowed). Its size gives the optical gain of each mode, `og`, where `z = og × residual`.
2. **R** = R_noise + R_aliasing, from the AI4AO model of the WFS (as in `DAOKalman.ipynb`), divided by og².
3. **Telemetry** under the integrator, and the pseudo open loop `pol = z / og + c(k − delay)`.
4. **The model.** VAR(N) by least squares, Q from its residuals, the steady-state Kalman gain (Riccati equation), and the prediction `A^delay`.
5. **The learnt predictor.** `var`, `K` and `C` are trained on the telemetry to minimise the residual. Then the robust step keeps it stable for optical-gain errors in `ROBUST_OG`.
6. **On the bench.** The integrator, the model and the learnt predictor, one after the other. They use the fixed-gain loop, split around the WFS read: one `N_KF × N_KF` product between reading `z` and sending the command.
7. **Second round.** The optical gain under the learnt predictor, which is higher than under the integrator because the residual is smaller. The predictor is retrained on its own telemetry with it.
8. **Save** everything.

The loop is open while the notebook computes (sections 2, 4, 5, 7). Each run starts with `WARM` frames of integrator, which close the loop again and settle the filter, before the predictor takes over.

**Conventions.** As in `DAOKalman.ipynb`, the command `c` is sent as `-c`, and the integrator is `c = leak c + gain z`. The notebook works with `zs = SIGN z`, where `SIGN = sign(gain)`, so the integrator is `c = leak c + |gain| zs` whatever the sign of the bench's gain.

**Without DAO**, `fake_dao.py` simulates the bench (AR(1) turbulence, WFS gain 0.7). It has no AI4AO calibration, so use `R_SOURCE = "scalar"`:

```
python fake_dao.py DAOVarNLQG.ipynb --wfs-gain 0.7 --lockstep --rate 5000 --set R_SOURCE='"scalar"'
```

In [ ]:
import os
# One thread per BLAS call: the per-frame products are small, and a thread pool would only add
# jitter. It must be set before numpy is imported (restart the kernel to change it).
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

import gc
import json
import sys
import time
from collections import deque
from pathlib import Path

import numpy as np
import torch
import pylab as plt

try:
    import dao
except ImportError:              # no DAO on this machine: simulated bench, see fake_dao.py
    import fake_dao
    bench = fake_dao.install(delay=2, sign=+1, wfs_gain=0.7, noise=0.05)
    import dao

# varn_lqg's model and learnt predictor. On the bench, copy that folder there and set the path.
VARN_DIR = Path.cwd().parent / "RamaCup" / "rama_controller" / "varn_lqg"
if not (VARN_DIR / "varn_model.py").exists():
    raise FileNotFoundError(f"{VARN_DIR}: set VARN_DIR to the varn_lqg folder")
sys.path.insert(0, str(VARN_DIR))
from varn_model import ahead, companion, fit_var, kalman_gain, split_delay
from learned_predictor import (filter_matrix, held_out, hidden_loop_radius, residual_rms,
                               robust_mix, train_predictor)

device = "cuda"

## Settings

In [ ]:
N_KF = 70                 # modes with the VARN-LQG; the modes above keep the bench integrator
ORDER = 2                 # N of the VAR(N) model
DELAY = 2.0               # loop delay (frames), measured in section 1; may be fractional

N_TELEMETRY = 10000       # integrator frames the model is fitted on
N_RUN = 10000             # frames of each run on the bench (after the WARM frames)
WARM = 500                # integrator frames at the start of each run
N_DITHER = 3000           # frames of dither, to measure the delay and the optical gain
DITHER_FRACTION = 0.2     # dither amplitude, as a fraction of the integrator's command RMS
U_MAX = 10.0              # safety: the loop opens if a command exceeds this

R_SOURCE = "wfs"          # "wfs": R_noise + R_aliasing from the AI4AO WFS model (as DAOKalman)
                          # "files": R_noise.npy + R_aliasing.npy in VARN_DIR (make_R.py)
                          # "scalar": R_SCALAR * I (var_lqg's assumption)
R_SCALE = 1.0             # R is multiplied by this
R_SCALAR = 0.3 / N_KF
DARE_N_ITER = 500         # Riccati iterations, at most

LEARN = "all"             # "all": var, K, C trained; "gain": K, C only (var from least squares)
LEARN_SECONDS = 60.0      # wall-clock budget of one training (the loop is open meanwhile)
TRAIN_THREADS = 4         # torch threads for the training
ROBUST_OG = (0.7, 1.0, 1.3)  # the learnt predictor stays stable for the optical gain off by these

## Shared memories and the loop

Same shared memories as `DAOKalman.ipynb`. `read_z` skips the frames that piled up in the semaphore while Python was busy (as in `DAOAdaptiveKalman.ipynb`).

`run_loop` runs one closed loop. With a predictor `(var, K, C)`, the filter is the steady-state Kalman filter with a fixed gain, on the pseudo open loop with its mean `mu` removed:

$$
\begin{aligned}
pol_k &= zs_k / og + (1-f)\,c_{k-n} + f\,c_{k-n-1} - \mu, & \text{delay} = n + f\\
x_k &= F x_{k-1} + K\,pol_k, & F = A - K\,var\\
u_k &= \mu + C x_k
\end{aligned}
$$

It is split around the WFS read, so that the command waits for only one product:

| when | what |
|---|---|
| before the read | `w = F x + K ((1-f) c[k-n] + f c[k-n-1] - mu)`, `b = mu + C w` |
| after the read | `u = b + (C K / og) zs`, the command, sent as `-u` |
| after sending | `x = w + (K / og) zs` |

The filter runs from the first frame, including the integrator's `WARM` frames, so its state has settled when the predictor takes over. The modes above `N_KF` keep the integrator throughout.

In [ ]:
MODAL_VECTOR_INFERENCE_SHM = dao.shm("/tmp/pyrModesNN.im.shm")    # WFS modal measurement z
CONTROL_SIGNAL_SHM = dao.shm("/tmp/ctrlModes07.im.shm")            # modal DM command (gets -c)
SHM_LOOP_CMD_NN_SHM = dao.shm("/tmp/lpCmdNN.im.shm")               # loop closed flag
SHM_LOOP_GAIN_SHM = dao.shm("/tmp/lpGain.im.shm")
SHM_LOOP_LEAK_SHM = dao.shm("/tmp/lpLeak.im.shm")

shm = dao.shm("/tmp/lpCmdNNModalEnable.im.shm")
shm.set_data(np.full_like(np.asarray(shm.get_data()), 0))

gain = float(SHM_LOOP_GAIN_SHM.get_data()[0, 0])
leak = float(SHM_LOOP_LEAK_SHM.get_data()[0, 0])
SIGN = float(np.sign(gain)) or 1.0       # zs = SIGN z: the integrator is c = leak c + |gain| zs
Nmodes = MODAL_VECTOR_INFERENCE_SHM.get_data().size
print(f"gain {gain}, leak {leak}, SIGN {SIGN:+.0f}, {Nmodes} measured modes, {N_KF} with VARN-LQG")


def read_z(min_wait=5e-5):
    """
    Wait for a new WFS frame and return a copy of it, flattened.

    If Python fell behind, frames piled up in the semaphore and a read returns at once, with a
    frame that was already processed. Those reads are skipped: keep reading until one actually
    waits (> min_wait seconds) for a new frame.
    """
    while True:
        t0 = time.perf_counter()
        z = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)
        if time.perf_counter() - t0 > min_wait:
            return np.array(z, dtype=np.float32).reshape(-1)


def run_loop(predictor=None, n_frames=None, warm=WARM, og=None, mu=None, dither=None, seed=0):
    """
    Closed loop on the bench: `warm` frames of integrator, then the predictor (if any).

    Parameters
    ----------
    predictor : tuple (var, K, C) or None
        var [n, N n], K [N n, n], C [n, N n] (varn_lqg's params), n = N_KF. None: the
        integrator throughout.
    n_frames : int
        Frames in all (warm included). Default WARM + N_RUN.
    og : ndarray [n] or None
        Optical gain of each mode (zs = og x residual). Default 1.
    mu : ndarray [n] or None
        Mean of pol, removed before the filter and added back to the command. Default 0.
    dither : ndarray [n] or None
        Amplitude of a random +-dither added to the commands after the warm frames.

    Returns
    -------
    log : dict
        z [frames, n] (zs, the first n modes), c [frames, n] (commands, the DM gets -c),
        p [frames, n] (dither), kalman [frames] (bool: the predictor was in control),
        t_crit [frames] (s, from the WFS read to the command sent), stopped (str), and on the
        fake bench `bench` (its first and last frame).
    """
    n_frames = WARM + N_RUN if n_frames is None else n_frames
    n = N_KF
    og = np.ones(n) if og is None else np.asarray(og, float)
    mu = np.zeros(n) if mu is None else np.asarray(mu, float)
    n_d, f_d = split_delay(DELAY)
    rng = np.random.default_rng(seed)
    if predictor is not None:
        var, K, C = (np.asarray(m, float) for m in predictor)
        F = filter_matrix(var, K)
        Kz = K / og                          # K (zs / og) = Kz zs
        CKz = C @ Kz                         # the only product after the WFS read
        x = np.zeros(F.shape[0])
    hist = deque([np.zeros(n)] * (n_d + 1), maxlen=n_d + 1)    # hist[-j] = c_{k-j}
    u_int, u_hi, g = np.zeros(n), np.zeros(Nmodes - n), abs(gain)
    log = {key: np.zeros((n_frames, n), np.float32) for key in ("z", "c", "p")}
    kalman, t_crit = np.zeros(n_frames, bool), np.zeros(n_frames)

    control = CONTROL_SIGNAL_SHM.get_data() * 0
    cs = control.reshape(-1)                 # flat view: cs[i] is mode i
    CONTROL_SIGNAL_SHM.set_data(control)
    b0 = len(dao.bench.residual) if hasattr(dao, "bench") else None
    stopped, done = "done", 0
    gc.disable()
    try:
        for k in range(n_frames):
            if not SHM_LOOP_CMD_NN_SHM.get_data()[0, 0]:
                stopped = f"frame {k}: loop opened from DAO"
                break
            use_kf = predictor is not None and k >= warm
            p = dither * rng.choice((-1.0, 1.0), n) if dither is not None and k >= warm else 0.0
            if predictor is not None:        # before the read
                applied = (1.0 - f_d) * hist[-n_d] + f_d * hist[-n_d - 1] - mu
                w = F @ x + K @ applied
                b = mu + C @ w

            z = read_z()
            t0 = time.perf_counter()
            zs = SIGN * z.astype(np.float64)
            zk = zs[:n]
            if use_kf:
                u = b + CKz @ zk
            else:
                u_int = leak * u_int + g * zk
                u = u_int
            c = u + p
            u_hi = leak * u_hi + g * zs[n:Nmodes]
            cs[:n] = -c
            cs[n:Nmodes] = -u_hi
            if np.abs(cs).max() > U_MAX:
                stopped = f"frame {k}: command above U_MAX"
                break
            CONTROL_SIGNAL_SHM.set_data(control)
            t_crit[k] = time.perf_counter() - t0

            if predictor is not None:        # after sending
                x = w + Kz @ zk
                if not np.all(np.isfinite(x)):
                    stopped = f"frame {k}: non-finite state"
                    break
            hist.append(c)
            log["z"][k], log["c"][k], log["p"][k], kalman[k] = zk, c, p, use_kf
            done = k + 1
    finally:
        gc.enable()
        CONTROL_SIGNAL_SHM.set_data(control * 0)
    log = {key: v[:done] for key, v in log.items()}
    log.update(kalman=kalman[:done], t_crit=t_crit[:done], stopped=stopped)
    if b0 is not None:
        log["bench"] = (b0, len(dao.bench.residual))
    if stopped != "done":
        print("stopped:", stopped)
    return log


def pseudo_open_loop(log, og, delay):
    """pol_k = zs_k / og + (1 - f) c_{k-n} + f c_{k-n-1}, for k = n + 1 ... (frames, n)."""
    n_d, f_d = split_delay(delay)
    z, c = log["z"].astype(float), log["c"].astype(float)
    s, T = n_d + 1, len(z)
    applied = (1.0 - f_d) * c[s - n_d:T - n_d] + f_d * c[s - n_d - 1:T - n_d - 1]
    return z[s:] / og + applied


def dither_response(log, max_lag=6):
    """The response of zs to the dither at each lag: in all, [lags], and per mode, [lags, n]."""
    k0 = int(np.flatnonzero(np.any(log["p"] != 0, axis=1))[0])
    Z, P = log["z"][k0:].astype(float), log["p"][k0:].astype(float)
    per_mode, total = [], []
    for j in range(max_lag + 1):
        zp, pp = Z[j:] * P[:len(P) - j], P[:len(P) - j] ** 2
        per_mode.append(zp.sum(0) / pp.sum(0))
        total.append(zp.sum() / pp.sum())
    return np.array(total), np.array(per_mode)


def delay_and_gain(total, per_mode):
    """
    The delay n + f and the optical gain per mode, from the dither response. The DM applies
    (1 - f) c_{k-n} + f c_{k-n-1}, so the response is -og (1 - f) at lag n and -og f at n + 1.

    Returns delay, the measured gain per mode, and the gain used (clipped to [0.5, 2] x median).
    """
    j = int(np.argmax(np.abs(total[:-1] + total[1:])))
    f = total[j + 1] / (total[j] + total[j + 1])
    n_d = j + 1 if f > 0.9 else j
    f = 0.0 if (f < 0.1 or f > 0.9) else round(float(f), 2)
    measured = -(per_mode[n_d] + (per_mode[n_d + 1] if f else 0.0))
    med = np.median(measured)
    if med <= 0:
        raise RuntimeError(f"the response to the dither has the wrong sign ({total.round(2)}): check SIGN")
    return n_d + f, measured, np.clip(measured, 0.5 * med, 2.0 * med)


def fs4_fraction(z):
    """Share of the power of z near fs/4: ringing there means the optical gain is underestimated."""
    S = np.abs(np.fft.rfft(z - z.mean(0), axis=0)) ** 2
    f = np.fft.rfftfreq(len(z))
    return S[(f > 0.2) & (f < 0.3)].sum() / S[1:].sum()


def compare(runs, og, settle=200):
    """Residual of each run, on its frames after WARM + settle, and the loop's compute time."""
    print(f"{'run':12s} {'frames':>7s} {'RMS zs':>9s} {'RMS zs/og':>10s} {'fs/4':>5s} "
          f"{'t median':>9s} {'t 99%':>7s} {'t max':>7s}" + ("  true residual" if hasattr(dao, "bench") else ""))
    for name, log in runs.items():
        z = log["z"][WARM + settle:].astype(float)
        if len(z) < 100:
            print(f"{name:12s} too short ({len(log['z'])} frames, {log['stopped']})")
            continue
        t = log["t_crit"][WARM + settle:] * 1e6
        line = (f"{name:12s} {len(z):7d} {np.sqrt((z ** 2).mean()):9.4g} "
                f"{np.sqrt(((z / og) ** 2).mean()):10.4g} {100 * fs4_fraction(z):4.0f}% "
                f"{np.median(t):7.0f}us {np.quantile(t, 0.99):5.0f}us {t.max():5.0f}us")
        if "bench" in log:
            b0, b1 = log["bench"]
            line += f"  {dao.bench.residual_rms(N_KF)[b0 + WARM + settle:b1].mean():.4g}"
        print(line)
    for name, log in runs.items():
        rms = np.sqrt((log["z"].astype(float) ** 2).mean(axis=1))
        plt.plot(np.convolve(rms, np.ones(100) / 100, mode="valid"), label=name)
    plt.axvline(WARM, color="k", ls=":")
    plt.xlabel("frame"); plt.ylabel("RMS of zs (100-frame mean)"); plt.yscale("log"); plt.legend(); plt.show()

## 1. Delay and optical gain

Integrator on all the modes, plus a random ±dither `p` on the first `N_KF` modes, after the `WARM` frames. The response of `zs` to the dither at lag `j`, $\sum zs_k\, p_{k-j} / \sum p^2$, is zero before the delay. At the delay `n + f` it is `-og (1 - f)` at lag `n` and `-og f` at lag `n + 1`. So the lags give the delay, and the sum of the two gives the optical gain of each mode.

`og` matters more than for the integrator. The controller rebuilds the turbulence as `zs / og + c`. If `og` is off by a factor ρ, part of its own command comes back in `pol`, a hidden loop that the robust step (section 5) has to cover.

In [ ]:
log0 = run_loop(None, 1000)                                   # integrator: the size of the commands
amp = DITHER_FRACTION * log0["c"][500:].std(0)
log_dither = run_loop(None, WARM + N_DITHER, dither=amp, seed=1)
total, per_mode = dither_response(log_dither)
DELAY, og_measured, OG = delay_and_gain(total, per_mode)
# DELAY = 2.0                                                 # to force a value

print("response per lag:", total.round(3))
print(f"delay = {DELAY:g} frames, optical gain: median {np.median(og_measured):.3f}, "
      f"10-90% [{np.quantile(og_measured, 0.1):.3f}, {np.quantile(og_measured, 0.9):.3f}]")
plt.plot(og_measured, ".", label="measured"); plt.plot(OG, "x", label="used")
plt.xlabel("mode"); plt.ylabel("optical gain"); plt.legend(); plt.show()

## 2. R = R_noise + R_aliasing

In the units of `zs`, then divided by `og_i og_j` for the pseudo open loop `zs / og`.

- `"wfs"` (as in `DAOKalman.ipynb`): R_noise from the photon and read-out noise through the reconstructor, and R_aliasing as the covariance of the modes reconstructed from residual phases with every DM mode removed. The WFS noise is off for the aliasing, so that it is not counted twice. About 1000 batches of 16 phases, a few minutes on the GPU.
- `"files"`: `R_noise.npy` and `R_aliasing.npy` from `make_R.py`.
- `"scalar"`: `R_SCALAR × I`.

R is computed for AI4AO's linear reconstructor. The bench's `pyrModesNN` comes from the CNN, so R is only an approximation of its noise. The learnt predictor (section 5) does not rely on R: it only starts from the model built with it.

In [ ]:
def to_np(t):
    return t.detach().cpu().double().numpy() if torch.is_tensor(t) else np.asarray(t, float)


if R_SOURCE == "wfs":
    import copy
    from AI4AO import DeformableMirror, PhaseDataset, PyramidWFS
    from AI4AO.paths import DATA_DIR
    from kalman.Kalman_tools_numpy import estimate_R
    from mmengine import Config
    from tqdm import tqdm

    PATH = DATA_DIR / "Data"
    PARAMFILE = r"/home/rama/rama-dev/foyarzun/CNN/Scripts/Rama_params.py"
    AtmosParams = Config.fromfile(PARAMFILE)["AtmosParams"]
    WFSParams = Config.fromfile(PARAMFILE)["WFSParams"]
    LoopParams = Config.fromfile(PARAMFILE)["LoopParams"]
    DMParams = Config.fromfile(PARAMFILE)["DMParams"]

    M2C = np.load(PATH / "M2C.npy").astype(np.float32)
    WFSParams["centralObstruction"] = 0.
    wfs = PyramidWFS(WFSParams, device)
    wfs.LoadCalibration(PATH / "RamaWFS.pth")
    wfs.eval()
    dm = DeformableMirror(WFSParams, DMParams, device)
    dm.LoadCalibration(PATH / "RamaDM.pth")
    modes = dm(torch.from_numpy(M2C.T).to(device))[:N_KF]
    wfs.modulation = 3
    wfs.BuildReconstructionMatrix(modes)

    R_noise = to_np(estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, wfs.Nphotons, wfs.RON))

    AtmosParams_aliasing = copy.copy(AtmosParams)
    AtmosParams_aliasing["Nphases"] = 16
    AtmosParams_aliasing["r0"] = [0.05, 0.2]
    LoopParams["levelOfCorrection"] = [1., 1.]
    dataset_aliasing = PhaseDataset(WFSParams, AtmosParams_aliasing, LoopParams, DMParams, device)
    dataset_aliasing.generateClosedLoop = True
    use_noise, wfs.useNoise = getattr(wfs, "useNoise", True), False      # no WFS noise here
    outputs = []
    with torch.no_grad():
        for _ in tqdm(range(1000)):
            batch = dataset_aliasing[0]
            outputs.append(wfs.GetReconstructedOPD(wfs(batch["opd"], batch["pupil"])))
    wfs.useNoise = use_noise
    R_aliasing = to_np(torch.cov(torch.stack(outputs).squeeze().reshape(-1, N_KF).T))
elif R_SOURCE == "files":
    R_noise = np.load(VARN_DIR / "R_noise.npy")[:N_KF, :N_KF].astype(float)
    R_aliasing = np.load(VARN_DIR / "R_aliasing.npy")[:N_KF, :N_KF].astype(float)
elif R_SOURCE == "scalar":
    R_noise, R_aliasing = R_SCALAR * np.eye(N_KF), np.zeros((N_KF, N_KF))
else:
    raise ValueError(f"R_SOURCE {R_SOURCE!r}: 'wfs', 'files' or 'scalar'")

R_zs = R_SCALE * (R_noise + R_aliasing)
R_zs = 0.5 * (R_zs + R_zs.T)


def R_for(og):
    """R in pseudo-open-loop units, R_zs / (og_i og_j), with its eigenvalues floored (positive definite)."""
    R = R_zs / np.outer(og, og)
    eigvals, eigvecs = np.linalg.eigh(R)
    floor = 1e-6 * max(float(np.mean(np.diag(R))), 1e-30)
    return (eigvecs * np.clip(eigvals, floor, None)) @ eigvecs.T


print(f"mean variance (zs units): R_noise {np.mean(np.diag(R_noise)):.3g}, R_aliasing {np.mean(np.diag(R_aliasing)):.3g}")

## 3. Telemetry under the integrator

`N_TELEMETRY` frames of integrator, and the pseudo open loop `pol = zs / og + c(k − delay)`, the turbulence as the controller rebuilds it. Its mean `mu` is removed for the fit, and the controller adds it back to its command.

In [ ]:
log_tel = run_loop(None, WARM + N_TELEMETRY)
pol = pseudo_open_loop(log_tel, OG, DELAY)[WARM:]
mu = pol.mean(0)
polc = pol - mu
res_integrator = log_tel["z"][WARM:].astype(float) / OG                # its residual, in pol units
print(f"{len(pol)} frames: RMS of pol {np.sqrt((polc ** 2).mean()):.4g}, "
      f"of the integrator's residual zs / og {np.sqrt((res_integrator ** 2).mean()):.4g}")

## 4. The model: VAR(N), Q, R, the Kalman gain and the prediction

$$pol_{t+1} = var\,[pol_t, \dots, pol_{t-N+1}] + e_{t+1}, \qquad Q = \mathrm{cov}(e)$$

The state is `x = [pol_t, ..., pol_t-N+1]`, with `A = [[var], [I 0]]` and `H = [I 0 ... 0]`. K is the steady-state Kalman gain of the Riccati equation. The prediction is `C`, the first `N_KF` rows of `A^delay`, which for a fractional delay is `(1 - f) A^n + f A^(n+1)`.

The residuals are scored on the held-out frames (the last 30 %), with the exact filter, in `pol` units. They include the WFS noise, which no controller can remove.

In [ ]:
n = N_KF
var, e_cov = fit_var(polc.T, ORDER)
A = companion(var)
Q = np.zeros_like(A)
Q[:n, :n] = e_cov
K, iterations = kalman_gain(A, Q, R_for(OG), DARE_N_ITER)
C = ahead(A, DELAY)[:n]
model = (var, K, C)

val = held_out(polc, DELAY)
print(f"VAR({ORDER}): |eig A| max {np.max(np.abs(np.linalg.eigvals(A))):.4f}, Riccati {iterations} it., "
      f"|eig F| max {np.max(np.abs(np.linalg.eigvals(filter_matrix(var, K)))):.4f}")
print("hidden loop radius for og x", ROBUST_OG, ":",
      [round(hidden_loop_radius(*model, DELAY, r), 3) for r in ROBUST_OG])
print(f"held-out residual RMS: no correction {np.sqrt((val[60:] ** 2).mean()):.4g}, integrator "
      f"{np.sqrt((res_integrator[-len(val):] ** 2).mean()):.4g}, model {residual_rms(val, *model, DELAY):.4g}")

## 5. The learnt predictor

`var`, `K` and `C` are trained together, starting from the model. The aim is the residual the controller leaves on the telemetry, `res_t = pol_t − (1 − f) p_{t−n} − f p_{t−n−1}`, where `p_k = C x_k` (see `RamaCup/LEARNED_PREDICTOR_TRAINING.md`). Training stops when the held-out residual stops improving, or after `LEARN_SECONDS`.

**Robust step.** Learning only for the residual pushes the predictor to the edge of the hidden loop's stability. So the predictor is moved back towards the model, `(1 − α) model + α learnt`, to the largest α for which that loop stays stable when the optical gain is off by every factor in `ROBUST_OG`.

In [ ]:
with torch.enable_grad():
    *learnt_raw, info = train_predictor(polc, *model, DELAY, train_var=(LEARN == "all"),
                                        max_seconds=LEARN_SECONDS, threads=TRAIN_THREADS)
learnt, alpha, worst = robust_mix(model, learnt_raw, DELAY, rhos=ROBUST_OG)

print(f"training: {info['iterations']} it., {info['seconds']:.1f} s, {info['taps']} taps, |eig F| {info['radius_F']:.4f}")
print(f"robust step: alpha {alpha:.2f}, worst hidden-loop radius {worst:.3f}")
for name, pr in (("model", model), ("learnt, no margin", learnt_raw), ("learnt, robust", learnt)):
    print(f"held-out residual RMS, {name:18s} {residual_rms(val, *pr, DELAY):.4g}")

## 6. On the bench: integrator, model and learnt

Each run has `WARM` integrator frames, then `N_RUN` frames with the controller (the integrator in the first run). The table compares the frames after `WARM + 200`:

- **RMS zs** is the residual as the WFS measures it. Under different controllers the optical gain is not the same, so the WFS sees a different fraction of the true residual.
- **RMS zs/og** uses the optical gain of section 1.
- **fs/4** is the share of the power of `zs` near a quarter of the frame rate. Ringing there means the optical gain is underestimated.
- **t** is the time from the WFS read to the command sent.

The bench's turbulence is not the same from one run to the next, so repeat the runs (or swap their order) before concluding.

In [ ]:
runs = {}
runs["integrator"] = run_loop(None)
runs["model"] = run_loop(model, og=OG, mu=mu)
runs["learnt"] = run_loop(learnt, og=OG, mu=mu)
compare(runs, OG)

## 7. Second round: the optical gain under the learnt predictor

The pyramid's optical gain rises when the residual falls, so under the learnt predictor it is higher than the `og` measured under the integrator, by ρ = og_true / og. This step measures it again with the same dither, on top of the learnt predictor. Then it rebuilds `pol` from the learnt run's telemetry with the new optical gain.

On that telemetry, the training starts from the better (held out) of a new model and the predictor in place, as the competition optimizer does from its second fit. It then gets the same robust step.

In [ ]:
if runs["learnt"]["stopped"] != "done":
    raise RuntimeError("the learnt run stopped early: " + runs["learnt"]["stopped"])

log_dither_k = run_loop(learnt, WARM + N_DITHER, og=OG, mu=mu, dither=amp, seed=2)
total_k, per_mode_k = dither_response(log_dither_k)
delay_k, og_measured_k, OG_K = delay_and_gain(total_k, per_mode_k)
print(f"under the learnt predictor: delay {delay_k:g}, optical gain median {np.median(og_measured_k):.3f}, "
      f"x {np.median(og_measured_k / og_measured):.2f} that under the integrator")
plt.plot(og_measured, ".", label="under the integrator"); plt.plot(og_measured_k, ".", label="under the learnt predictor")
plt.xlabel("mode"); plt.ylabel("optical gain"); plt.legend(); plt.show()

# the learnt run's telemetry, with the optical gain of its regime
log_k = {key: runs["learnt"][key][WARM + 200:] for key in ("z", "c")}
pol2 = pseudo_open_loop(log_k, OG_K, DELAY)
mu2 = pol2.mean(0)
polc2 = pol2 - mu2

var2, e_cov2 = fit_var(polc2.T, ORDER)
A2 = companion(var2)
Q2 = np.zeros_like(A2)
Q2[:n, :n] = e_cov2
K2, _ = kalman_gain(A2, Q2, R_for(OG_K), DARE_N_ITER)
model2 = (var2, K2, ahead(A2, DELAY)[:n])

val2 = held_out(polc2, DELAY)
rms_model2, rms_in_place = residual_rms(val2, *model2, DELAY), residual_rms(val2, *learnt, DELAY)
start2, start_name = (model2, "new model") if rms_model2 < rms_in_place else (learnt, "in place")
with torch.enable_grad():
    *learnt2_raw, info2 = train_predictor(polc2, *start2, DELAY, train_var=(LEARN == "all"),
                                          max_seconds=LEARN_SECONDS, threads=TRAIN_THREADS)
learnt2, alpha2, worst2 = robust_mix(start2, learnt2_raw, DELAY, rhos=ROBUST_OG)
print(f"held-out residual RMS: new model {rms_model2:.4g}, in place {rms_in_place:.4g}, learnt from "
      f"{start_name} {residual_rms(val2, *learnt2, DELAY):.4g} ({info2['iterations']} it., "
      f"robust step {alpha2:.2f}, worst radius {worst2:.3f})")

runs["learnt 2"] = run_loop(learnt2, og=OG_K, mu=mu2)
compare(runs, OG)

## 8. Save

Everything needed to replay the runs offline: the settings, the delay and optical gains, R, the predictors, and each run's telemetry (`zs`, the commands `c`, sent as `-c`, the dither, the compute times).

In [ ]:
settings = {k: globals()[k] for k in ("N_KF", "ORDER", "DELAY", "N_TELEMETRY", "N_RUN", "WARM", "N_DITHER",
                                      "DITHER_FRACTION", "U_MAX", "R_SOURCE", "R_SCALE", "LEARN",
                                      "LEARN_SECONDS", "SIGN", "gain", "leak")}
settings["ROBUST_OG"] = list(ROBUST_OG)
save = dict(settings=json.dumps(settings), OG=OG, og_measured=og_measured, R_noise=R_noise,
            R_aliasing=R_aliasing, mu=mu, dither_amp=amp, telemetry_z=log_tel["z"], telemetry_c=log_tel["c"])
for name, pr in (("model", model), ("learnt", learnt)):
    save.update({f"{name}_var": pr[0], f"{name}_K": pr[1], f"{name}_C": pr[2]})
if "learnt 2" in runs:
    save.update(OG_K=OG_K, og_measured_k=og_measured_k, mu2=mu2,
                learnt2_var=learnt2[0], learnt2_K=learnt2[1], learnt2_C=learnt2[2])
for name, log in runs.items():
    key = name.replace(" ", "")
    save.update({f"run_{key}_{k}": v for k, v in log.items() if isinstance(v, np.ndarray)})
    save[f"run_{key}_stopped"] = log["stopped"]

telemetry_file = f"varn_lqg_bench_{time.strftime('%Y%m%d_%H%M%S')}.npz"
np.savez(telemetry_file, **save)
print("saved", telemetry_file)
if hasattr(dao, "bench"):        # fake bench only
    print(dao.bench.summary(n_modes=N_KF))